# Recreate SHAP summary plot from `shap_values_for_sharing.pkl`

This notebook loads the shared SHAP bundle and recreates the SHAP summary plot using the same plotting logic as the original notebook.

**Expected file in the same folder as this notebook:**
- `shap_values_for_sharing.pkl`

If the file lives elsewhere, update `PKL_PATH` in the first code cell.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# Path to the shared SHAP bundle
PKL_PATH = "../results/figures/visualizations_cv/shap_manuscript/shap_values_for_sharing.pkl"

# Where to save outputs
OUTPUT_DIR = "../results/figures/visualizations_cv/shap_manuscript"

print(f"Using SHAP bundle: {PKL_PATH}")
print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")

In [ ]:
# Load shared SHAP bundle
with open(PKL_PATH, "rb") as f:
    shap_data = pickle.load(f)

# Unpack objects
shap_values_oof = shap_data["shap_values"]
X_shap_oof = shap_data["feature_data"]
feature_names = shap_data.get("feature_names", list(X_shap_oof.columns))
base_value = shap_data.get("base_value", None)
categorical_color_maps = shap_data.get("categorical_color_maps", {})
feature_display_names = shap_data.get("feature_display_names", {})

model_info = shap_data.get("model_info", {})
RANDOM_STATE = model_info.get("random_state", 42)

# Ensure expected formats
if isinstance(X_shap_oof, np.ndarray):
    X_shap_oof = pd.DataFrame(X_shap_oof, columns=feature_names)

if isinstance(shap_values_oof, pd.DataFrame):
    shap_values_oof = shap_values_oof.values

print("Loaded SHAP bundle successfully.")
print(f"  SHAP values shape: {np.shape(shap_values_oof)}")
print(f"  Feature data shape: {X_shap_oof.shape}")
print(f"  Random state: {RANDOM_STATE}")
print(f"  Base value: {base_value}")
print(f"  Number of categorical color maps: {len(categorical_color_maps)}")
print(f"  Number of display names: {len(feature_display_names)}")

In [ ]:
# Recompute SHAP importance rankings
mean_abs_shap = np.abs(shap_values_oof).mean(axis=0)
shap_importance_df = pd.DataFrame({
    "feature": X_shap_oof.columns,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

print("Top 20 features by mean |SHAP|:")
display(shap_importance_df.head(20))

In [ ]:
def create_shap_summary_plot(xlim=None, suffix="", top_n=12):
    """
    Create publication-quality SHAP summary plot

    Parameters
    ----------
    xlim : tuple or None
        (xmin, xmax) for x-axis limits. If None, uses automatic limits
    suffix : str
        Suffix to add to filename (e.g., "_zoomed")
    top_n : int
        Number of top features to display (default 12)
    """
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D

    # Use top N features ranked by mean |SHAP| from shap_importance_df
    top_features = shap_importance_df.head(top_n)["feature"].tolist()
    # Filter to only features present in X_shap_oof
    top_features = [f for f in top_features if f in X_shap_oof.columns]
    top_idx = [X_shap_oof.columns.get_loc(f) for f in top_features]

    fig, ax = plt.subplots(figsize=(12, 9), facecolor="white", dpi=150)
    ax.set_facecolor("white")
    np.random.seed(RANDOM_STATE)

    vertical_spacing = 1.6

    for i, feature in enumerate(reversed(top_features)):
        original_idx = len(top_features) - 1 - i
        feature_idx = top_idx[original_idx]

        shap_vals_feature = shap_values_oof[:, feature_idx]
        feature_data = X_shap_oof[feature].values

        y_base = i * vertical_spacing
        y_positions = y_base + np.random.normal(0, 0.07, len(shap_vals_feature))

        point_size = 40
        point_alpha = 0.6

        if feature in categorical_color_maps:
            cat_map = categorical_color_maps[feature]
            unique_values = pd.unique(feature_data)

            for value in unique_values:
                if pd.isna(value):
                    color = cat_map.get("MISSING", "#95a5a6")
                    mask = pd.isna(feature_data)
                else:
                    color = cat_map.get(value)
                    if color is None and isinstance(value, str):
                        color = cat_map.get(value.strip())
                    if color is None and isinstance(value, str):
                        color = cat_map.get(value.lower())
                    if color is None:
                        color = "#95a5a6"
                    mask = feature_data == value

                if mask.any():
                    ax.scatter(
                        shap_vals_feature[mask],
                        y_positions[mask],
                        c=color,
                        alpha=point_alpha,
                        s=point_size,
                        edgecolors="white",
                        linewidths=0.3,
                    )

        elif X_shap_oof[feature].dtype in ["object", "category"]:
            # Categorical but no custom color map — use Set2
            unique_vals = [v for v in pd.unique(feature_data) if pd.notna(v)]
            colors_map = plt.cm.Set2(np.linspace(0, 1, max(len(unique_vals), 1)))
            for idx, val in enumerate(unique_vals):
                mask = feature_data == val
                if mask.any():
                    ax.scatter(
                        shap_vals_feature[mask],
                        y_positions[mask],
                        c=[colors_map[idx]],
                        alpha=point_alpha,
                        s=point_size,
                        edgecolors="white",
                        linewidths=0.3,
                    )
        else:
            # Continuous features
            try:
                if feature_data.dtype == "object":
                    feature_data_numeric = pd.to_numeric(feature_data, errors="coerce")
                else:
                    feature_data_numeric = feature_data

                ax.scatter(
                    shap_vals_feature,
                    y_positions,
                    c=feature_data_numeric,
                    cmap="coolwarm",
                    alpha=point_alpha,
                    s=point_size,
                    vmin=np.nanpercentile(feature_data_numeric, 5),
                    vmax=np.nanpercentile(feature_data_numeric, 95),
                    edgecolors="white",
                    linewidths=0.3,
                )
            except Exception:
                ax.scatter(
                    shap_vals_feature,
                    y_positions,
                    c="#95a5a6",
                    alpha=point_alpha,
                    s=point_size,
                    edgecolors="white",
                    linewidths=0.3,
                )

    # Y-axis labels using display names
    y_tick_positions = [i * vertical_spacing for i in range(len(top_features))]
    display_labels = [feature_display_names.get(f, f) for f in reversed(top_features)]
    ax.set_yticks(y_tick_positions)
    ax.set_yticklabels(display_labels, fontsize=14)

    ax.set_xlabel("SHAP Value", fontsize=16, fontweight="bold")
    ax.axvline(x=0, color="#333333", linestyle="-", linewidth=2, zorder=0)
    ax.grid(False)

    for spine in ["top", "right", "bottom", "left"]:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_linewidth(1.5)
        ax.spines[spine].set_color("#333333")

    if xlim is not None:
        ax.set_xlim(xlim)

    ax.set_ylim(-0.8, (len(top_features) - 1) * vertical_spacing + 0.8)
    ax.tick_params(
        axis="both",
        which="major",
        labelsize=14,
        width=1.5,
        length=6,
        colors="#333333",
    )

    # ============================================================
    # LEGEND
    # ============================================================
    legend_elements = []

    # Continuous gradient
    legend_elements.append(
        Line2D([0], [0], marker="", color="none", label="Continuous Features:", markersize=0)
    )
    gradient_colors = [plt.cm.coolwarm(v) for v in [0.0, 0.25, 0.5, 0.75, 1.0]]
    gradient_labels = ["  Low", "", "  Mid", "", "  High"]
    for color, label in zip(gradient_colors, gradient_labels):
        legend_elements.append(
            Patch(facecolor=color, edgecolor="#333333", linewidth=0.5, label=label)
        )

    legend_elements.append(
        Line2D([0], [0], marker="", color="none", label="", markersize=0)
    )

    # last.EJC
    legend_elements.append(
        Line2D([0], [0], marker="", color="none", label="last.EJC:", markersize=0)
    )
    legend_elements.append(
        Patch(facecolor="#e74c3c", edgecolor="#333333", linewidth=1.2, label="  last.exon")
    )
    legend_elements.append(
        Patch(facecolor="#3498db", edgecolor="#333333", linewidth=1.2, label="  upstream")
    )
    legend_elements.append(
        Patch(
            facecolor="#2ecc71",
            edgecolor="#333333",
            linewidth=1.2,
            label="  penultimate.last50bp",
        )
    )
    legend_elements.append(
        Line2D([0], [0], marker="", color="none", label="", markersize=0)
    )

    # first.200 / first.100 (categorical TRUE/FALSE)
    legend_elements.append(
        Line2D([0], [0], marker="", color="none", label="PTC in first 200nt:", markersize=0)
    )
    legend_elements.append(
        Patch(facecolor="#9b59b6", edgecolor="#333333", linewidth=1.2, label="  Yes")
    )
    legend_elements.append(
        Patch(facecolor="#f39c12", edgecolor="#333333", linewidth=1.2, label="  No")
    )

    legend = ax.legend(
        handles=legend_elements,
        loc="upper left",
        bbox_to_anchor=(1.05, 1.0),
        frameon=True,
        fontsize=13,
        edgecolor="#333333",
        fancybox=False,
        framealpha=1.0,
        facecolor="white",
        labelspacing=0.5,
    )
    legend.get_frame().set_linewidth(1.5)

    for text in legend.get_texts():
        if ":" in text.get_text() and not text.get_text().startswith("  "):
            text.set_fontweight("bold")

    plt.tight_layout()

    output_files = [
        (f"Fig_SHAP_summary_top{top_n}{suffix}.png", {"dpi": 300, "format": "png"}),
        (f"Fig_SHAP_summary_top{top_n}{suffix}.pdf", {"format": "pdf"}),
        (
            f"Fig_SHAP_summary_top{top_n}{suffix}.tiff",
            {"dpi": 300, "format": "tiff", "pil_kwargs": {"compression": "tiff_lzw"}},
        ),
    ]

    for filename, save_kwargs in output_files:
        plt.savefig(
            os.path.join(OUTPUT_DIR, filename),
            bbox_inches="tight",
            facecolor="white",
            edgecolor="none",
            **save_kwargs,
        )

    print(f"\n✅ Saved publication figure with top {top_n} features")
    print(f"   Output: {OUTPUT_DIR}/Fig_SHAP_summary_top{top_n}{suffix}.[png/pdf/tiff]")

    plt.show()
    plt.close()

    return fig

In [ ]:
# Main figure
create_shap_summary_plot(xlim=None, suffix="", top_n=12)

# Optional zoomed version:
# create_shap_summary_plot(xlim=(-0.25, 0.25), suffix="_zoomed", top_n=12)